- ingest data from volume to delta table
- add metadata column such as file_name , ingest_timestamp

In [0]:
%run ../00-common/01.environment_config

In [0]:
source_file_path = f"{landing_file_path}/circuits.csv"
full_table_name = f"{catalog_name}.{bronze_schema}.circuit"


In [0]:
from pyspark.sql.types import StructField,StructType,DoubleType,StringType
# for production scenarion we define schema explicitly if new data is also come then also no issue occur, for development purpose we can used .option("inferSchema","true") this option
 
circuit_schema = StructType([
    StructField("circuitId", StringType()),
    StructField("url", StringType()),
    StructField("circuitName", StringType()),
    StructField("lat", DoubleType()),
    StructField("long", DoubleType()),
    StructField("locality", StringType()),
    StructField("country", StringType())
])

In [0]:
df_curcuit = spark.read.format("csv") \
            .option("header","true") \
            .schema(circuit_schema) \
            .load(source_file_path)


In [0]:
from pyspark.sql import functions as F

df_curcuit_final = df_curcuit.withColumn("ingestion_date", current_timestamp()) \
                            .withColumn("source_file",col("_metadata.file_path"))

display(df_curcuit_final)


In [0]:
df_curcuit_final.write.format("delta") \
                        .mode("overwrite") \
                        .saveAsTable(full_table_name)

In [0]:
%sql
select * from formula1.bronze.circuit